# EXPRESSÕES REGULARES E FUNÇÕES

Limpeza de dados + funções reutilizáveis

**objetivos**:

- Limpar caracteres
- Padronizar
- Verificar critérios



## Ler a base de clientes e fazer o diagnóstico inicial

In [1]:
from google.colab import files
uploaded = files.upload()

Saving clientes_sujos.csv to clientes_sujos.csv
Saving pedidos_loja.csv to pedidos_loja.csv
Saving pedidos_loja.json to pedidos_loja.json
Saving pedidos_loja.xlsx to pedidos_loja.xlsx


In [2]:
import pandas as pd
import re

clientes = pd.read_csv(
    "clientes_sujos.csv",
    dtype={
        "telefone": "string",
        "cpf": "string",
        "email": "string"
    }
)

In [3]:
print("Tamanho da tabela: ", clientes.shape)

clientes.info()

print(clientes.isna().sum())

display(
    clientes[["nome", "telefone", "email", "cpf"]]
)

Tamanho da tabela:  (8, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   cliente_id     8 non-null      int64 
 1   nome           8 non-null      object
 2   telefone       8 non-null      string
 3   email          8 non-null      string
 4   cpf            8 non-null      string
 5   data_cadastro  8 non-null      object
 6   observacao     8 non-null      object
dtypes: int64(1), object(3), string(3)
memory usage: 580.0+ bytes
cliente_id       0
nome             0
telefone         0
email            0
cpf              0
data_cadastro    0
observacao       0
dtype: int64


,nome,telefone,email,cpf
0,Ana Souza,(21) 98888-1111,ana.souza@email.com,123.456.789-10
1,Bruno Lima,21 97777 2222,bruno.lima@email,98765432100
2,Carla Dias,+55 (21) 96666-3333,carla.dias@empresa.com.br,111.222.333-44
3,Diego Rocha,(11)95555-4444,diego_rocha@email.com,22233344455
4,Eva Martins,21944445555,eva.martins@email.com,333.444.555-66
5,Felipe Nunes,sem telefone,felipe.nunes@email.com,444.555.666-77
6,Giovana Reis,(31) 93333-7777,giovana.reis@email.com,55566677788
7,Hugo Alves,31-92222-8888,hugo.alves@empresa,666.777.888-99


# Introdução regex com um único telefone



In [4]:
texto = "(21) 98888-1111"

somente_numeros = re.sub(r"\D", "", texto)
print(somente_numeros)

21988881111


In [5]:
cpf = "123.456.789-10"

cpf_limpo = re.sub(r"\D", "", cpf)
print(cpf_limpo)

12345678910


# Transformar a limpeza em uma função

In [6]:
def limpar_numero(texto):
  if pd.isna(texto):
    return ""

  texto = str(texto)

  return re.sub(r"\D", "", texto)

In [7]:
telefone = limpar_numero(texto="(21) 98888-1111")
print(telefone)

21988881111


In [8]:
print(limpar_numero("123.456.789"))
print(limpar_numero("sem telefone"))
print(limpar_numero("012.345.678-90"))

123456789

01234567890


In [9]:
clientes["telefone_limpo"] = clientes["telefone"].apply(limpar_numero)

clientes["cpf_limpo"] = clientes["cpf"].apply(limpar_numero)

In [10]:
display(
    clientes[
        ["nome", "telefone", "telefone_limpo", "cpf", "cpf_limpo"]
    ]
)

,nome,telefone,telefone_limpo,cpf,cpf_limpo
0,Ana Souza,(21) 98888-1111,21988881111,123.456.789-10,12345678910
1,Bruno Lima,21 97777 2222,21977772222,98765432100,98765432100
2,Carla Dias,+55 (21) 96666-3333,5521966663333,111.222.333-44,11122233344
3,Diego Rocha,(11)95555-4444,11955554444,22233344455,22233344455
4,Eva Martins,21944445555,21944445555,333.444.555-66,33344455566
5,Felipe Nunes,sem telefone,,444.555.666-77,44455566677
6,Giovana Reis,(31) 93333-7777,31933337777,55566677788,55566677788
7,Hugo Alves,31-92222-8888,31922228888,666.777.888-99,66677788899


# Introdução de lambda

para limpar os espaços dos emails.

In [11]:
emails_para_limpar = clientes["email"].fillna("")

clientes["email_limpo"] = emails_para_limpar.apply(
    lambda texto: str(texto).strip()
)

"ana.souza@email.com "
"ana.souza@email.com"

In [12]:
display(
    clientes[
        ["email", "email_limpo"]
    ]
)

,email,email_limpo
0,ana.souza@email.com,ana.souza@email.com
1,bruno.lima@email,bruno.lima@email
2,carla.dias@empresa.com.br,carla.dias@empresa.com.br
3,diego_rocha@email.com,diego_rocha@email.com
4,eva.martins@email.com,eva.martins@email.com
5,felipe.nunes@email.com,felipe.nunes@email.com
6,giovana.reis@email.com,giovana.reis@email.com
7,hugo.alves@empresa,hugo.alves@empresa


# Verificar o formato dos emails com uma função

In [13]:
def email_valido(email):
  if pd.isna(email):
    return False

  email = str(email).strip()

  padrao = r"^[\w\.-]+@[\w\.-]+\.\w+$"

  return bool(re.match(padrao, email))

In [14]:
print(email_valido("ana.souza@email.com"))
print(email_valido("bruno.lima@email"))
print(email_valido("carla.dias@empresa.com.br"))

True
False
True


In [15]:
clientes["email_formato_ok"] = clientes["email_limpo"].apply(email_valido)

In [16]:
display(
    clientes[
        ["nome", "email", "email_limpo", "email_formato_ok"]
    ]
)

,nome,email,email_limpo,email_formato_ok
0,Ana Souza,ana.souza@email.com,ana.souza@email.com,True
1,Bruno Lima,bruno.lima@email,bruno.lima@email,False
2,Carla Dias,carla.dias@empresa.com.br,carla.dias@empresa.com.br,True
3,Diego Rocha,diego_rocha@email.com,diego_rocha@email.com,True
4,Eva Martins,eva.martins@email.com,eva.martins@email.com,True
5,Felipe Nunes,felipe.nunes@email.com,felipe.nunes@email.com,True
6,Giovana Reis,giovana.reis@email.com,giovana.reis@email.com,True
7,Hugo Alves,hugo.alves@empresa,hugo.alves@empresa,False


Pergunta: Retirar espaços de um email garante que ele esteja correto?

In [17]:
# Registros que não passaram

display(
    clientes.loc[
        ~clientes["email_formato_ok"],
        ["nome", "email_limpo"]
    ]
)

,nome,email_limpo
1,Bruno Lima,bruno.lima@email
7,Hugo Alves,hugo.alves@empresa


In [18]:
print(clientes.head())  #texto monoespaçado, como no  terminal

display(clientes.head()) #Tabela HTML formatada e alinhada

   cliente_id         nome             telefone                      email  \
0           1    Ana Souza      (21) 98888-1111       ana.souza@email.com    
1           2   Bruno Lima        21 97777 2222           bruno.lima@email   
2           3   Carla Dias  +55 (21) 96666-3333  carla.dias@empresa.com.br   
3           4  Diego Rocha       (11)95555-4444      diego_rocha@email.com   
4           5  Eva Martins          21944445555     eva.martins@email.com    

              cpf data_cadastro                                  observacao  \
0  123.456.789-10    01/09/2026  Cliente VIP; prefere contato por whatsapp.   
1     98765432100    2026-09-02                            email incompleto   
2  111.222.333-44    03-09-2026                             comprou 2 vezes   
3     22233344455    2026/09/04                         telefone sem espaço   
4  333.444.555-66    05/09/2026                                 cadastro ok   

  telefone_limpo    cpf_limpo                email_limpo

,cliente_id,nome,telefone,email,cpf,data_cadastro,observacao,telefone_limpo,cpf_limpo,email_limpo,email_formato_ok
0,1,Ana Souza,(21) 98888-1111,ana.souza@email.com,123.456.789-10,01/09/2026,Cliente VIP; prefere contato por whatsapp.,21988881111,12345678910,ana.souza@email.com,True
1,2,Bruno Lima,21 97777 2222,bruno.lima@email,98765432100,2026-09-02,email incompleto,21977772222,98765432100,bruno.lima@email,False
2,3,Carla Dias,+55 (21) 96666-3333,carla.dias@empresa.com.br,111.222.333-44,03-09-2026,comprou 2 vezes,5521966663333,11122233344,carla.dias@empresa.com.br,True
3,4,Diego Rocha,(11)95555-4444,diego_rocha@email.com,22233344455,2026/09/04,telefone sem espaço,11955554444,22233344455,diego_rocha@email.com,True
4,5,Eva Martins,21944445555,eva.martins@email.com,333.444.555-66,05/09/2026,cadastro ok,21944445555,33344455566,eva.martins@email.com,True


# Completando as verificações de telefone e CPF

In [19]:
# Critérios de Tamanho

clientes["telefone_tamanho_ok"] = clientes["telefone_limpo"].apply(
    lambda texto: len(texto) in [10, 11]
)

clientes["cpf_11_digitos"] = clientes["cpf_limpo"].apply(
    lambda texto: len(texto) == 11
)

In [20]:
display(
    clientes[
        ["nome",
         "telefone",
         "telefone_limpo",
         "telefone_tamanho_ok",
         "cpf_limpo",
         "cpf_11_digitos"]
    ]
)

,nome,telefone,telefone_limpo,telefone_tamanho_ok,cpf_limpo,cpf_11_digitos
0,Ana Souza,(21) 98888-1111,21988881111,True,12345678910,True
1,Bruno Lima,21 97777 2222,21977772222,True,98765432100,True
2,Carla Dias,+55 (21) 96666-3333,5521966663333,False,11122233344,True
3,Diego Rocha,(11)95555-4444,11955554444,True,22233344455,True
4,Eva Martins,21944445555,21944445555,True,33344455566,True
5,Felipe Nunes,sem telefone,,False,44455566677,True
6,Giovana Reis,(31) 93333-7777,31933337777,True,55566677788,True
7,Hugo Alves,31-92222-8888,31922228888,True,66677788899,True


| Cliente | Resultado da limpeza | Interpretação |
|---|---|---|
| Carla Dias | `5521966663333` | O número inclui `55`, o código do país, e ficou com 13 dígitos |
| Felipe Nunes | Texto vazio | O conteúdo original era `sem telefone`, sem nenhum dígito |

Os dois resultados ficaram como "False", porém os dois clientes não apresentam o mesmo problema.

A Carla apresenta um formato que a regra que fizemos de tamanho não contempla. Já o Felipe não apresenta um número.

**Decisão**: preservar os registros e indicar o resultado das verificações e depois fazer uma regra para normalizar o código do país.

# Modularização

## Funções, módulos e importação

**objetivos**
- Organizar código em funções.
- Separar as nossas responsabilidades.
- Criar e importar um módulo Python.
- Montar um pipeline simples de leitura, limpeza, análise e exportação

In [21]:
telefone = limpar_numero("(21) 98888-1111")

print(telefone)

21988881111


| Conceito | Neste exemplo |
|---|---|
| Nome da função | `limpar_numero` |
| Parâmetro | `texto`, na definição da função |
| Argumento | `"(21) 98888-1111"`, passado na chamada |
| Retorno | `"21988881111"` |
| Variável que recebe o retorno | `telefone` |

In [22]:
print(limpar_numero("012.345.678-90"))
print(limpar_numero("sem telefone"))

01234567890



# Guardaremos o as funções de limpeza em um arquivo chamado limpeza.py. Esse arquivo será nosso módulo."

# Criar o módulo no Colab de modularização

In [23]:
%%writefile limpeza.py
import re
import pandas as pd

def limpar_numero(texto):
  if pd.isna(texto):
    return ""

  texto = str(texto)

  return re.sub(r"\D", "", texto)

def email_valido(email):
  if pd.isna(email):
    return False

  email = str(email).strip()

  padrao = r"^[\w\.-]+@[\w\.-]+\.\w+$"

  return bool(re.match(padrao, email))

def tratar_clientes(df):
    resultado = df.copy()

    resultado["telefone_limpo"] = resultado["telefone"].apply(
        limpar_numero
    )

    resultado["cpf_limpo"] = resultado["cpf"].apply(
        limpar_numero
    )

    resultado["email_limpo"] = resultado["email"].fillna("").apply(
        lambda texto: str(texto).strip()
    )

    resultado["email_formato_ok"] = resultado["email_limpo"].apply(
        email_valido
    )

    resultado["telefone_tamanho_ok"] = resultado["telefone_limpo"].apply(
        lambda texto: len(texto) in [10, 11]
    )

    resultado["cpf_11_digitos"] = resultado["cpf_limpo"].apply(
        lambda texto: len(texto) == 11
    )

    return resultado

Writing limpeza.py


In [26]:
from google.colab import files

uploaded = files.upload()

Saving clientes_sujos.csv to clientes_sujos (2).csv
Saving pedidos_loja.csv to pedidos_loja (1).csv
Saving pedidos_loja.json to pedidos_loja (1).json
Saving pedidos_loja.xlsx to pedidos_loja (1).xlsx


## Vamos indicar qual arquivo será lido

In [28]:
import pandas as pd
from limpeza import tratar_clientes

arquivo_clientes = "clientes_sujos (2).csv"

clientes_brutos = pd.read_csv(
    arquivo_clientes,
    dtype={
        "telefone": "string",
        "cpf": "string",
        "email": "string"
    }
)

clientes_tratados = tratar_clientes(clientes_brutos)

display(clientes_tratados.head())

,cliente_id,nome,telefone,email,cpf,data_cadastro,observacao,telefone_limpo,cpf_limpo,email_limpo,email_formato_ok,telefone_tamanho_ok,cpf_11_digitos
0,1,Ana Souza,(21) 98888-1111,ana.souza@email.com,123.456.789-10,01/09/2026,Cliente VIP; prefere contato por whatsapp.,21988881111,12345678910,ana.souza@email.com,True,True,True
1,2,Bruno Lima,21 97777 2222,bruno.lima@email,98765432100,2026-09-02,email incompleto,21977772222,98765432100,bruno.lima@email,False,True,True
2,3,Carla Dias,+55 (21) 96666-3333,carla.dias@empresa.com.br,111.222.333-44,03-09-2026,comprou 2 vezes,5521966663333,11122233344,carla.dias@empresa.com.br,True,False,True
3,4,Diego Rocha,(11)95555-4444,diego_rocha@email.com,22233344455,2026/09/04,telefone sem espaço,11955554444,22233344455,diego_rocha@email.com,True,True,True
4,5,Eva Martins,21944445555,eva.martins@email.com,333.444.555-66,05/09/2026,cadastro ok,21944445555,33344455566,eva.martins@email.com,True,True,True


In [29]:
# Verificação rápida

print("email_limpo" in clientes_brutos.columns)
print("email_limpo" in clientes_tratados.columns)

False
True


# Criar uma função para o relatório

Produzir um resumo para o responsável pelos cadastros em que ele consiga interpretar

In [30]:
def gerar_relatorio_clientes(df):
  return {
      "total_clientes": len(df),

      "emails_com_formato_ok": int(df["email_formato_ok"].sum()),

      "telefones_com_tamanho_ok": int(
          df["telefone_tamanho_ok"].sum()),

      "cpfs_com_11_digitos": int(df["cpf_11_digitos"].sum())

  }

relatorio = gerar_relatorio_clientes(clientes_tratados)

print(relatorio)

{'total_clientes': 8, 'emails_com_formato_ok': 6, 'telefones_com_tamanho_ok': 6, 'cpfs_com_11_digitos': 8}


## Salvando tabela

In [31]:
clientes_tratados.to_csv(
    "clientes_tratados_final.csv",
    index=False
)

In [32]:
import json

with open(
    "relatorio_clientes.json",
    "w",
    encoding="utf-8"
) as arquivo:
    json.dump(
        relatorio,
        arquivo,
        ensure_ascii=False,
        indent=4
    )

## Pergunta

Para investigar o cadastro da cliente Carla, qual arquivo devemos consultar?

Arquivo CSV.

Baixando as entregas no colab conforme necessário



In [33]:
files.download("clientes_tratados_final.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [34]:
files.download("relatorio_clientes.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [35]:
files.download("limpeza.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Organizar o pipeline

Pipeline é uma sequência de etapas encadeadas, em que cada saída de cada etapa vira a entrada da próxima

CSV bruto → tratar_clientes() → gerar-relatorio_clientes() → resumo final




In [36]:
import pandas as pd
import json
from limpeza import tratar_clientes


# 1. Ler a base
clientes_brutos = pd.read_csv(
    "clientes_sujos (2).csv",
    dtype={
        "telefone": "string",
        "cpf": "string",
        "email": "string"
    }
)


# 2. Tratar os registros
clientes_tratados = tratar_clientes(clientes_brutos)


# 3. Calcular os indicadores
relatorio = gerar_relatorio_clientes(clientes_tratados)


# 4. Salvar a tabela
clientes_tratados.to_csv(
    "clientes_tratados_final.csv",
    index=False
)


# 5. Salvar o relatório
with open(
    "relatorio_clientes.json",
    "w",
    encoding="utf-8"
) as arquivo:
    json.dump(
        relatorio,
        arquivo,
        ensure_ascii=False,
        indent=4
    )


# 6. Apresentar o resumo
print(relatorio)

{'total_clientes': 8, 'emails_com_formato_ok': 6, 'telefones_com_tamanho_ok': 6, 'cpfs_com_11_digitos': 8}


# Mini-projeto - Cadastro limpo

1. Quantos pedidos foram entregues?

2. Qual foi o valor total vendido nos pedidos entregues?

3. Qual produto aparece mais vezes?

4. Qual filtro foi usado para chegar à resposta?

A empresa precisa receber uma base de clientes tratada e um resumo dos cadastros. Utilizando o módulo para ler dados limpeza.py, vocês vão aplicar regras, gerar indicadores e salvar os arquivos e escrever uma solução.

In [ ]:
email_formato_ok
telefone_tamanho_ok
cpf_11_digitos

In [37]:
from google.colab import files
uploaded = files.upload()

Saving clientes_sujos.csv to clientes_sujos (3).csv


In [38]:
# Saber quais arquivos foram enviados
print(list(uploaded.keys()))

['clientes_sujos (3).csv']


In [42]:
import pandas as pd
import json
from limpeza import tratar_clientes

def gerar_relatorio_clientes(df):
  return {
      "total_clientes": len(df),

      "emails_com_formato_ok": int(df["email_formato_ok"].sum()),

      "telefones_com_tamanho_ok": int(
          df["telefone_tamanho_ok"].sum()
      ),

      "cpfs_com_11_digitos": int(
          df["cpf_11_digitos"].sum()
      )
  }

## Ler a base

In [43]:
clientes_brutos = pd.read_csv(
    "clientes_sujos (3).csv",
    dtype={
        "telefone": "string",
        "cpf": "string",
        "email": "string"
    }
)

## Aplicar o tratamento

In [44]:
clientes_tratados = tratar_clientes(clientes_brutos)

In [45]:
# Gerar relatório

relatorio = gerar_relatorio_clientes(clientes_tratados)

## Salvar a tabela tratada

In [53]:
clientes_tratados.to_csv(
    "clientes_tratados_final_3.csv",
    index=False
)

## Salvar o relatório


In [49]:
with open(
    "relatorio_clientes_3.json",
    "w",
    encoding="utf-8"
) as arquivo:
  json.dump(
      relatorio,
      arquivo,
      ensure_ascii=False,
      indent=4
  )

## Apresentando os resultados

In [50]:
print(relatorio)

{'total_clientes': 8, 'emails_com_formato_ok': 6, 'telefones_com_tamanho_ok': 6, 'cpfs_com_11_digitos': 8}


In [51]:
# Total de cadastros de clientes analisados

print(
    f"Foram analisados {relatorio['total_clientes']} cadastros."
)

Foram analisados 8 cadastros.


In [54]:
# Quais emails passaram na verificação de formato

print(
    f"{relatorio['emails_com_formato_ok']} emails passaram na verificação simplificada de formato"
)

6 emails passaram na verificação simplificada de formato


In [56]:
# Quais telefones passaram na veificação

print(
    f"{relatorio["telefones_com_tamanho_ok"]} telefones passaram na verificação de tamanho"
)

6 telefones passaram na verificação de tamanho


In [59]:
print(
    f"{relatorio['cpfs_com_11_digitos']} CPFs apresentaram "
    "11 dígitos após a limpeza."
)

8 CPFs apresentaram 11 dígitos após a limpeza.
